# 🌾 KisaanSathi: MobileNetV2 Transfer Learning on Kaggle GPU
### Crop Disease Classification & Explainable AI Pipeline

This notebook trains the industrial-grade **MobileNetV2** model on the **PlantVillage (38 Classes)** dataset using **Kaggle GPU acceleration (T4 x2 or P100)**.

### 📌 Key Features in this Notebook:
1. **Strict GPU Verification**: Auto-detects Kaggle GPU and stops with clear instructions if GPU is not turned on.
2. **Direct Google Drive Import OR Kaggle Dataset**: Connect your existing Google Drive processed folder directly without re-uploading from your PC.
3. **Zero Dataset Leakage**: Uses the existing pre-split dataset (No regeneration; `Background_without_leaves` excluded).
4. **Exact 38 Classes**: Verifies all classes and image counts before compiling.
5. **High Performance `tf.data`**: Prefetching with `AUTOTUNE` on high-speed Kaggle SSD storage.
6. **Two-Stage Transfer Learning**:
   - **Stage 1 (Feature Extraction)**: Base MobileNetV2 frozen, Adam $lr = 10^{-3}$, 8 epochs.
   - **Stage 2 (Fine-Tuning)**: Top 30 layers unfreezed (BatchNormalization frozen), Adam $lr = 10^{-5}$, 8 epochs.
7. **Resumable & Safe Checkpointing**: Saves checkpoints after **every single epoch**; resumes automatically if session restarts.
8. **Independent Test Set Evaluation**: Real evaluation on unseen `test/` images (Accuracy, Precision, Recall, F1, Confusion Matrix, and Per-Class breakdown).
9. **Artifact Export & 1-Click Download**: Saves `crop_disease_model.keras`, `class_names.json`, and all diagnostic reports directly to `/kaggle/working/` for simple download.

---  
## ⚙️ STEP 1: Detect Kaggle GPU (Hardware Check)
This cell verifies that Kaggle GPU acceleration is active. If GPU is not enabled, it stops with a helpful message.

In [ ]:
# STEP 1: GPU Hardware Detection
import os
import sys
import tensorflow as tf

print("Python version:    ", sys.version.split()[0])
print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')

if not gpus:
    raise RuntimeError(
        "\n" + "="*72 + "\n"
        "❌ ERROR: NO GPU DETECTED IN THIS KAGGLE SESSION!\n\n"
        "To enable Kaggle GPU:\n"
        "1. Look at the right-hand sidebar menu in Kaggle.\n"
        "2. Click 'Settings' (the gear icon on the top right of notebook editor).\n"
        "3. Under 'Accelerator', change 'None' to 'GPU T4 x2' (or 'GPU P100').\n"
        "4. Ensure 'Internet' toggle is set to 'Internet on'.\n"
        "5. Wait a few seconds for the Kaggle kernel to restart with GPU enabled.\n"
        "6. Re-run this cell!\n"
        + "="*72
    )

print(f"\n✅ SUCCESS: {len(gpus)} GPU(s) available for training!")
for i, gpu in enumerate(gpus):
    print(f"   - Device {i}: {gpu.name}")

# Display GPU hardware specs
os.system('nvidia-smi')

---  
## 📥 STEP 2A (OPTIONAL): Import Dataset from Google Drive into Kaggle
**If your data is already attached under `/kaggle/input`, you can SKIP this cell!**  
If your dataset is only in Google Drive, run this cell to download directly from your Google Drive link into Kaggle with **zero PC re-uploading**.

In [ ]:
# STEP 2A: Direct Download from Google Drive (Run only if dataset is NOT already in /kaggle/input)
# Your public Google Drive folder link:
GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1tB6vsKRB7dVv7LmeNmy-vSDT82qyqtay?usp=sharing"

import os
from pathlib import Path

# Check if dataset is already present before downloading
has_data = any((Path(p) / "train").exists() for p in ["/kaggle/input", "/kaggle/working/data", "/kaggle/working"] if Path(p).exists())

if not has_data:
    print("Installing gdown for direct Google Drive transfer...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "gdown"], check=False)
    
    os.makedirs("/kaggle/working/data", exist_ok=True)
    print("\nDownloading dataset directly from Google Drive into Kaggle...")
    # Download folder directly using folder ID
    os.system('gdown --folder "1tB6vsKRB7dVv7LmeNmy-vSDT82qyqtay" -O /kaggle/working/data')
    print("\n✅ Google Drive download completed!")
else:
    print("✅ Dataset is already present! Skipping Google Drive download.")

---  
## 📂 STEP 2B: Automatic Dataset Discovery & 38-Class Verification
This cell dynamically scans `/kaggle/input` and `/kaggle/working` to locate your `train`, `val`, and `test` folders. It guarantees:
- Exactly **38 crop classes** are detected.
- `Background_without_leaves` is **strictly excluded**.
- Train, validation, and test image counts are checked.

In [ ]:
# STEP 2B: Locate Dataset and Verify Split Directories
import os
from pathlib import Path

# Search /kaggle/input and /kaggle/working
search_roots = [Path("/kaggle/input"), Path("/kaggle/working")]
print("Searching for prepared dataset...")

train_dir, val_dir, test_dir = None, None, None
dataset_root = None

for root_path in search_roots:
    if not root_path.exists():
        continue
    for cand in root_path.rglob("train"):
        parent = cand.parent
        if (parent / "val").is_dir() and (parent / "test").is_dir():
            train_dir = parent / "train"
            val_dir = parent / "val"
            test_dir = parent / "test"
            dataset_root = parent
            break
    if train_dir is not None:
        break

if train_dir is None:
    raise FileNotFoundError(
        "\n" + "="*72 + "\n"
        "❌ ERROR: Could not locate 'train', 'val', and 'test' folders!\n\n"
        "You can supply the dataset using either:\n"
        "OPTION A: Run STEP 2A to download directly from your Google Drive link.\n"
        "OPTION B: Click '+ Add Input' -> 'Upload Dataset' -> attach your processed dataset.\n"
        + "="*72
    )

print(f"✅ Successfully located dataset at: {dataset_root}")
print(f"   - Train Directory:      {train_dir}")
print(f"   - Validation Directory: {val_dir}")
print(f"   - Test Directory:       {test_dir}")

# Enumerate and verify classes (strictly excluding Background_without_leaves)
EXCLUDE_CLASS = "Background_without_leaves"

train_classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir() and d.name != EXCLUDE_CLASS])
val_classes   = sorted([d.name for d in val_dir.iterdir()   if d.is_dir() and d.name != EXCLUDE_CLASS])
test_classes  = sorted([d.name for d in test_dir.iterdir()  if d.is_dir() and d.name != EXCLUDE_CLASS])

if EXCLUDE_CLASS in [d.name for d in train_dir.iterdir() if d.is_dir()]:
    print(f"⚠️ NOTE: Detected '{EXCLUDE_CLASS}' in dataset — automatically EXCLUDED from training classes!")

# Verify exactly 38 classes
if len(train_classes) != 38:
    raise ValueError(f"❌ Expected exactly 38 classes, but found {len(train_classes)} in train folder!")

if train_classes != val_classes or train_classes != test_classes:
    raise ValueError("❌ Class categories do not match identically across train, val, and test folders!")

print(f"\n✅ Verified exactly {len(train_classes)} usable crop disease & healthy classes.")

# Count valid images
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
train_count = sum(1 for p in train_dir.rglob("*") if p.suffix.lower() in VALID_EXTENSIONS and p.parent.name != EXCLUDE_CLASS)
val_count   = sum(1 for p in val_dir.rglob("*")   if p.suffix.lower() in VALID_EXTENSIONS and p.parent.name != EXCLUDE_CLASS)
test_count  = sum(1 for p in test_dir.rglob("*")  if p.suffix.lower() in VALID_EXTENSIONS and p.parent.name != EXCLUDE_CLASS)
total_images = train_count + val_count + test_count

print("\n📊 Dataset Image Count Summary:")
print(f"   - Training Set:   {train_count:,} images ({train_count/total_images*100:.1f}%)")
print(f"   - Validation Set: {val_count:,} images ({val_count/total_images*100:.1f}%)")
print(f"   - Test Set:       {test_count:,} images ({test_count/total_images*100:.1f}%)")
print(f"   - Total Usable:   {total_images:,} images")

---  
## 🚀 STEP 3: Optimized `tf.data` Loading & Class Mapping Export
Loads images at **$224 \times 224$** resolution with prefetching for peak GPU throughput, and saves `class_names.json` to the output artifacts folder.

In [ ]:
# STEP 3: Build tf.data Input Pipelines
import json

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

# Create working output directories
ARTIFACTS_DIR = Path("/kaggle/working/artifacts")
CHECKPOINTS_DIR = Path("/kaggle/working/checkpoints")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print("Building tf.data pipelines...")

train_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir),
    class_names=train_classes,  # Strict class ordering & exclusion
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(val_dir),
    class_names=train_classes,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    str(test_dir),
    class_names=train_classes,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

# Apply prefetching for optimal GPU saturation
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.prefetch(buffer_size=AUTOTUNE)

# Save ordered class mapping (0 to 37)
class_mapping = {str(i): name for i, name in enumerate(train_classes)}
class_names_path = ARTIFACTS_DIR / "class_names.json"
with open(class_names_path, "w", encoding="utf-8") as f:
    json.dump(class_mapping, f, indent=2)

print(f"✅ Successfully saved ordered class mapping to: {class_names_path}")
print(f"Sample classes: {[train_classes[i] for i in [0, 10, 20, 30]]}")

---  
## 🏗️ STEP 4: Build MobileNetV2 Architecture
Matches the KisaanSathi inference contract:
- **Training-only Augmentation**: Random flip, slight rotation, zoom, translation.
- **Normalization**: Rescaling $(pixel / 127.5) - 1.0$ to $[-1, 1]$.
- **Pre-trained Backbone**: MobileNetV2 with ImageNet weights.
- **Head**: GlobalAveragePooling2D ➔ Dropout(0.3) ➔ Dense(38, softmax).

In [ ]:
# STEP 4: Build Model Architecture
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# 1. Real-time data augmentation (active only during training)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

# 2. Pretrained MobileNetV2 Base (frozen for Stage 1)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

# 3. Assembly
inputs = layers.Input(shape=(224, 224, 3), name="input_image")
x = data_augmentation(inputs)
# Normalization to [-1, 1] range required by MobileNetV2
x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="mobilenet_rescaling")(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dropout(0.3, name="dropout")(x)
outputs = layers.Dense(len(train_classes), activation="softmax", name="predictions")(x)

model = models.Model(inputs, outputs, name="KisaanSathi_MobileNetV2")
print("Model summary:")
model.summary()

---  
## 🔄 STEP 5: Resumable Training & Checkpointing Setup
This cell checks if previous checkpoints exist in `/kaggle/working/checkpoints/`.
If your session ever disconnects or stops, re-running this cell will restore your latest trained weights automatically!

In [ ]:
# STEP 5: Checkpoint Detection & Resumable Session Loader
import glob

def find_latest_checkpoint():
    """Finds the most recent checkpoint file if resuming after disconnection."""
    # Check Stage 2 checkpoints first
    s2_files = sorted(glob.glob(str(CHECKPOINTS_DIR / "stage2_epoch_*.keras")))
    if s2_files:
        return s2_files[-1], "stage2"
    # Check Stage 1 checkpoints
    s1_files = sorted(glob.glob(str(CHECKPOINTS_DIR / "stage1_epoch_*.keras")))
    if s1_files:
        return s1_files[-1], "stage1"
    if (CHECKPOINTS_DIR / "stage1_best.keras").exists():
        return str(CHECKPOINTS_DIR / "stage1_best.keras"), "stage1_best"
    return None, None

latest_ckpt, ckpt_stage = find_latest_checkpoint()
if latest_ckpt:
    print(f"💡 Found existing checkpoint: {latest_ckpt} (Stage: {ckpt_stage})")
    print("To resume training from this checkpoint, uncomment the line below:")
    print(f"# model.load_weights('{latest_ckpt}')")
else:
    print("✅ No previous checkpoints found. Starting a fresh training run.")

---  
## 🎯 STEP 6: STAGE 1 Training (Feature Extraction - Frozen Base)
- **Base Model**: Frozen
- **Optimizer**: Adam with learning rate $= 10^{-3}$ ($0.001$)
- **Epochs**: 8 epochs
- **Checkpoint Policy**: Saves after **every epoch** + saves best validation accuracy checkpoint.
- **EarlyStopping & ReduceLROnPlateau**: Active to prevent overfitting.

In [ ]:
# STEP 6: Execute STAGE 1 Training
import pandas as pd

STAGE1_EPOCHS = 8

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_stage1 = [
    # 1. Save checkpoint after EVERY single epoch for full resilience
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINTS_DIR / "stage1_epoch_{epoch:02d}.keras"),
        save_weights_only=False,
        save_best_only=False,
        verbose=1
    ),
    # 2. Save best validation accuracy model
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINTS_DIR / "stage1_best.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    # 3. Dynamic LR reduction on plateau
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    # 4. Early stopping
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    # 5. Log all training numbers to CSV
    tf.keras.callbacks.CSVLogger(str(ARTIFACTS_DIR / "stage1_training_log.csv"))
]

print("="*72)
print("🚀 STARTING STAGE 1: Feature Extraction (MobileNetV2 Base Frozen)")
print("="*72)

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE1_EPOCHS,
    callbacks=callbacks_stage1
)

print("\n✅ STAGE 1 TRAINING COMPLETE!")

---  
## 🔬 STEP 7: STAGE 2 Training (Fine-Tuning Top 30 Layers)
- **Unfreeze**: Top 30 convolutional layers of MobileNetV2 (BatchNormalization layers remain frozen).
- **Optimizer**: Adam with delicate learning rate $= 10^{-5}$ ($0.00001$).
- **Epochs**: 8 epochs
- **Saves Best Model Directly**: Saves `crop_disease_model.keras` into the final artifacts directory.

In [ ]:
# STEP 7: Execute STAGE 2 Fine-Tuning
STAGE2_EPOCHS = 8

# Unfreeze base model
base_model.trainable = True

# Fine-tune only the top 30 layers, keep BatchNormalization frozen
unfreeze_layers = 30
for layer in base_model.layers[:-unfreeze_layers]:
    layer.trainable = False
for layer in base_model.layers[-unfreeze_layers:]:
    if isinstance(layer, layers.BatchNormalization) or "BatchNormalization" in layer.__class__.__name__:
        layer.trainable = False
    else:
        layer.trainable = True
trainable_count = sum(len(l.trainable_weights) for l in model.layers)
print(f"Fine-tuning active with top {unfreeze_layers} layers unfreezed ({trainable_count} trainable weight tensors).")

# Recompile with lower learning rate for stable fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

FINAL_MODEL_PATH = ARTIFACTS_DIR / "crop_disease_model.keras"

callbacks_stage2 = [
    # 1. Save checkpoint after EVERY single epoch
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINTS_DIR / "stage2_epoch_{epoch:02d}.keras"),
        save_weights_only=False,
        save_best_only=False,
        verbose=1
    ),
    # 2. Save best overall model to final artifact path
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FINAL_MODEL_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    # 3. Dynamic LR reduction on plateau
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    # 4. Early stopping
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    # 5. Log all training numbers to CSV
    tf.keras.callbacks.CSVLogger(str(ARTIFACTS_DIR / "stage2_training_log.csv"))
]

print("="*72)
print("🔬 STARTING STAGE 2: Fine-Tuning Top 30 Layers (Learning Rate: 1e-5)")
print("="*72)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE2_EPOCHS,
    callbacks=callbacks_stage2
)

# Ensure final model is saved to artifacts and root working directory
model.save(str(FINAL_MODEL_PATH))
model.save("/kaggle/working/crop_disease_model.keras")
print(f"\n✅ STAGE 2 COMPLETE! Final model saved to: {FINAL_MODEL_PATH}")

---  
## 📊 STEP 8: Independent Test Set Evaluation (Real Metrics)
Evaluates on the **never-before-seen** `test` split (5,497 images) and generates:
- Overall Accuracy, Loss
- Precision, Recall, F1-Score (Macro and Weighted)
- Per-Class Metrics Table
- High-resolution Confusion Matrix heatmap

In [ ]:
# STEP 8: Comprehensive Evaluation on Unseen Test Set
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

print("="*72)
print("🧪 EVALUATING ON INDEPENDENT TEST SET (Never seen during training/val)")
print("="*72)

test_loss, test_acc = model.evaluate(test_ds, verbose=1)

# Collect all true labels and predictions
print("Computing predictions across all test batches...")
y_true_indices = []
y_pred_indices = []

for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true_indices.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_indices.extend(np.argmax(preds, axis=1))

y_true_indices = np.array(y_true_indices)
y_pred_indices = np.array(y_pred_indices)

# Calculate macro and weighted metrics
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true_indices, y_pred_indices, average='macro', zero_division=0
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true_indices, y_pred_indices, average='weighted', zero_division=0
)

print("\n" + "="*72)
print(f"🎯 FINAL TEST SET ACCURACY:    {test_acc * 100:.2f}%")
print(f"🎯 FINAL TEST SET LOSS:        {test_loss:.4f}")
print(f"🎯 MACRO F1-SCORE:             {f1_macro:.4f}")
print(f"🎯 WEIGHTED F1-SCORE:          {f1_weighted:.4f}")
print(f"🎯 WEIGHTED PRECISION:         {precision_weighted:.4f}")
print(f"🎯 WEIGHTED RECALL:            {recall_weighted:.4f}")
print("="*72)

# Generate & save Classification Report
report_dict = classification_report(y_true_indices, y_pred_indices, target_names=train_classes, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_csv_path = ARTIFACTS_DIR / "per_class_metrics.csv"
report_df.to_csv(report_csv_path)
print(f"\n✅ Saved per-class metrics to: {report_csv_path}")

# Print classification report summary
print("\nDetailed Classification Report:")
print(classification_report(y_true_indices, y_pred_indices, target_names=train_classes, zero_division=0))

# Generate Confusion Matrix
cm = confusion_matrix(y_true_indices, y_pred_indices)
plt.figure(figsize=(18, 14))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Greens)
plt.title(f"Confusion Matrix (38 Classes) - Test Accuracy: {test_acc*100:.2f}%", fontsize=16)
plt.colorbar()
tick_marks = np.arange(len(train_classes))
plt.xticks(tick_marks, train_classes, rotation=90, fontsize=8)
plt.yticks(tick_marks, train_classes, fontsize=8)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()

cm_plot_path = ARTIFACTS_DIR / "confusion_matrix.png"
plt.savefig(cm_plot_path, dpi=300)
plt.show()
print(f"✅ Saved confusion matrix heatmap to: {cm_plot_path}")

# Save overall summary JSON
test_summary = {
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "macro_precision": float(precision_macro),
    "macro_recall": float(recall_macro),
    "macro_f1": float(f1_macro),
    "weighted_precision": float(precision_weighted),
    "weighted_recall": float(recall_weighted),
    "weighted_f1": float(f1_weighted),
    "total_test_images": int(len(y_true_indices))
}
with open(ARTIFACTS_DIR / "test_summary.json", "w", encoding="utf-8") as f:
    json.dump(test_summary, f, indent=2)
print(f"✅ Saved test summary to: {ARTIFACTS_DIR / 'test_summary.json'}")

---  
## 📈 STEP 9: Plot Training Curves & History
Visualizes accuracy and loss progression across both Stage 1 and Stage 2.

In [ ]:
# STEP 9: Plot and Save Training Curves
def extract_metric(h1, h2, metric):
    vals = []
    if h1 and metric in h1.history:
        vals.extend(h1.history[metric])
    if h2 and metric in h2.history:
        vals.extend(h2.history[metric])
    return vals

acc = extract_metric(history_stage1, history_stage2, 'accuracy')
val_acc = extract_metric(history_stage1, history_stage2, 'val_accuracy')
loss = extract_metric(history_stage1, history_stage2, 'loss')
val_loss = extract_metric(history_stage1, history_stage2, 'val_loss')

plt.figure(figsize=(14, 5))

# Accuracy subplot
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy', color='#2b8a3e', lw=2)
plt.plot(val_acc, label='Validation Accuracy', color='#1971c2', lw=2)
plt.axvline(x=len(history_stage1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('Training & Validation Accuracy', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# Loss subplot
plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', color='#c92a2a', lw=2)
plt.plot(val_loss, label='Validation Loss', color='#e67700', lw=2)
plt.axvline(x=len(history_stage1.history['loss'])-1, color='gray', linestyle='--', label='Fine-Tuning Start')
plt.title('Training & Validation Loss', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

curves_path = ARTIFACTS_DIR / "training_curves.png"
plt.savefig(curves_path, dpi=300)
plt.show()
print(f"✅ Saved training curves plot to: {curves_path}")

---  
## 📦 STEP 10: Package Artifacts for 1-Click Kaggle Download
Bundles all model artifacts and evaluation metrics into a single zip file in `/kaggle/working/`.

In [ ]:
# STEP 10: Bundle All Artifacts into Downloadable Archive
import shutil

ZIP_DEST = "/kaggle/working/kisaansathi_model_artifacts"
shutil.make_archive(ZIP_DEST, 'zip', str(ARTIFACTS_DIR))

print("="*72)
print("🎉 ALL TRAINING & EVALUATION TASKS ARE COMPLETE!")
print("="*72)
print("\n📁 Downloadable Files available in Kaggle 'Output' tab:")
print(f"   1. /kaggle/working/artifacts/crop_disease_model.keras  (Trained Model)")
print(f"   2. /kaggle/working/artifacts/class_names.json          (38 Class Mapping)")
print(f"   3. /kaggle/working/artifacts/test_summary.json         (Accuracy/Loss Summary)")
print(f"   4. /kaggle/working/artifacts/per_class_metrics.csv     (Per-Class Precision/Recall/F1)")
print(f"   5. /kaggle/working/artifacts/confusion_matrix.png      (Confusion Matrix Heatmap)")
print(f"   6. /kaggle/working/artifacts/training_curves.png       (Accuracy/Loss Progression)")
print(f"   7. /kaggle/working/kisaansathi_model_artifacts.zip     (Complete Zip Package)")
print("\n📥 HOW TO DOWNLOAD FROM KAGGLE:")
print("1. Look at the right sidebar under the 'Output' pane.")
print("2. Click the three dots (...) next to 'kisaansathi_model_artifacts.zip' or 'crop_disease_model.keras'.")
print("3. Click 'Download'.")
print("4. On your computer, extract/place 'crop_disease_model.keras' into:")
print("   c:\\Users\\ram03\\KisaanSathi\\ml\\models\\")